# 02 — Preprocessing & Label Conversion
Class filtering, stratified subsetting, train/val/test split, BDD100K → YOLO format conversion, and visual verification.

In [1]:
from pathlib import Path

_here = Path.cwd().resolve()
for _root in (_here, *_here.parents):
    if(_root / "dataset").is_dir() and (_root / "src").is_dir():
        DATASET_DIR = _root / "dataset"
        print(f"Dataset directory found: {DATASET_DIR}")
        break
else:
    raise FileNotFoundError("Dataset directory not found. Please ensure the 'dataset' directory exists in the root of the repository.")

Dataset directory found: C:\Users\micha\Downloads\Object-Detection-main\dataset


In [2]:
!pip install ultralytics --no-deps

In [3]:
import sys
import numpy as np

project_path = _root
if str(project_path) not in sys.path:
    sys.path.insert(0, str(project_path))

from src.utils import seed_everything, log_environment, CLASS_MAP, CLASS_NAMES

seed_everything()
log_environment()

PyTorch: 2.11.0+cu128
Ultralytics: 8.4.42
GPU: NVIDIA GeForce RTX 5090
CUDA: 12.8


In [4]:
import json
import os
import glob
import random


DATASET_PATH = str(_root / "dataset")
TRAIN_IMAGE_DIR = os.path.join(DATASET_PATH, "bdd100k_images_100k", "100k", "train")
VAL_IMAGE_DIR = os.path.join(DATASET_PATH, "bdd100k_images_100k", "100k", "val")
TRAIN_JSON = os.path.join(DATASET_PATH, "Bdd100k", "bdd100k_labels_images_train.json")
VAL_JSON = os.path.join(DATASET_PATH, "Bdd100k", "bdd100k_labels_images_val.json")
WORKING_ROOT = str(_root / "outputs" / "bdd100k_preprocessing")
OUTPUT_DIR = os.path.join(WORKING_ROOT, "bdd100k-yolo-subset-v1")
LABEL_OUTPUT_DIR = os.path.join(WORKING_ROOT, "yolo_labels_temp")
RESULTS_PLOTS = os.path.join(WORKING_ROOT, "results", "plots")
os.makedirs(WORKING_ROOT, exist_ok=True)
os.makedirs(RESULTS_PLOTS, exist_ok=True)

print("DATASET_PATH:", DATASET_PATH)
print("WORKING_ROOT:", WORKING_ROOT)
print("Done")

DATASET_PATH: C:\Users\micha\Downloads\Object-Detection-main\dataset
WORKING_ROOT: C:\Users\micha\Downloads\Object-Detection-main\outputs\bdd100k_preprocessing
Done


## Step 1: Load Annotations

In [5]:
from pathlib import Path

from src.bdd100k_io import load_bdd100k_train_val

_image_root = Path(TRAIN_IMAGE_DIR).parent
train_annotations, val_annotations, _bdd_meta = load_bdd100k_train_val(
    Path(DATASET_PATH),
    image_root=_image_root,
)
print(f"Annotation source: {_bdd_meta['source']}")

all_annotations = train_annotations + val_annotations
print(f"Total annotations loaded: {len(all_annotations)}")

val labels: 100%|██████████| 10000/10000 [00:00<00:00, 11059.54file/s]

Annotation source: per_image_json
Total annotations loaded: 80000


## Step 2: Class Filtering
Remove images that contain no instances of our 8 selected classes.

In [6]:
from src.preprocess import filter_relevant_images

filtered = filter_relevant_images(all_annotations)

Kept 80000 / 80000 images (100.0%) — dropped 0 with no selected-class labels


## Step 3: Stratified Subsetting
Sample a subset preserving weather × timeofday proportions.

In [7]:
from src.preprocess import stratified_subset

TRAIN_SIZE = 7000
VAL_SIZE = 1500
TEST_SIZE = 1500

train_items, val_items, test_items = stratified_subset(
    filtered, train_size=TRAIN_SIZE, val_size=VAL_SIZE, test_size=TEST_SIZE
)

Subset sizes — Train: 7000, Val: 1500, Test: 1500


## Step 4: Convert BDD100K Labels → YOLO Format

In [8]:
import shutil
import os

# 1. Clear the label output directory
if os.path.exists(LABEL_OUTPUT_DIR):
    print(f"Cleaning up labels at: {LABEL_OUTPUT_DIR}")
    shutil.rmtree(LABEL_OUTPUT_DIR)
os.makedirs(LABEL_OUTPUT_DIR, exist_ok=True)

old_jsons = [
    os.path.join(WORKING_ROOT, "train_annotations.json"),
    os.path.join(WORKING_ROOT, "val_annotations.json"),
    os.path.join(WORKING_ROOT, "test_annotations.json"),
    os.path.join(WORKING_ROOT, "filtered_annotations.json"),
]

for json_file in old_jsons:
    if os.path.exists(json_file):
        os.remove(json_file)

print("Cleanup complete. New Fresh Run")

Cleanup complete. New Fresh Run


In [9]:
import importlib

import src.convert_labels as _bdd_convert_labels

# Reload so edits to `src/convert_labels.py` apply without restarting the kernel.
importlib.reload(_bdd_convert_labels)
convert_bdd100k_to_yolo = _bdd_convert_labels.convert_bdd100k_to_yolo

print("convert_labels module:", _bdd_convert_labels.__file__)

# Ensure output folder exists (safe if you run this cell before the paths/plots cell).
os.makedirs(WORKING_ROOT, exist_ok=True)

all_items = train_items + val_items + test_items

filtered_json_path = os.path.join(WORKING_ROOT, "filtered_annotations.json")
with open(filtered_json_path, "w") as f:
    json.dump(all_items, f)

_bdd_img_roots = [os.path.abspath(TRAIN_IMAGE_DIR), os.path.abspath(VAL_IMAGE_DIR)]
print("BDD100K image search roots (conversion + Step 5 copy):")
for _r in _bdd_img_roots:
    print(" ", _r, "exists=" + str(os.path.isdir(_r)))

if train_items:
    _probe = _bdd_convert_labels._find_bdd100k_image(train_items[0].get("name", ""), _bdd_img_roots)
    print("Probe first train item:", repr(train_items[0].get("name")), "->", _probe)
    if _probe is None:
        raise FileNotFoundError(
            "First train item did not resolve to an image under the roots above. "
            "Confirm BDD100K images are under bdd100k_images_100k/100k/train and .../val."
        )

for split_name, items in [("train", train_items), ("val", val_items), ("test", test_items)]:
    split_json = os.path.join(WORKING_ROOT, f"{split_name}_annotations.json")
    with open(split_json, "w") as f:
        json.dump(items, f)

    print(f"\nConverting {split_name} split...")
    convert_bdd100k_to_yolo(
        json_path=split_json,
        image_dirs=_bdd_img_roots,
        output_dir=os.path.join(LABEL_OUTPUT_DIR, split_name),
    )

convert_labels module: C:\Users\micha\Downloads\Object-Detection-main\src\convert_labels.py
BDD100K image search roots (conversion + Step 5 copy):
  C:\Users\micha\Downloads\Object-Detection-main\dataset\bdd100k_images_100k\100k\train exists=True
  C:\Users\micha\Downloads\Object-Detection-main\dataset\bdd100k_images_100k\100k\val exists=True
Probe first train item: '7554131a-6fa923df' -> C:\Users\micha\Downloads\Object-Detection-main\dataset\bdd100k_images_100k\100k\train\7554131a-6fa923df.jpg

Converting train split...
Converted: 7000
Skipped (no image file): 0
Skipped (no selected-class labels): 0

Converting val split...
Converted: 1500
Skipped (no image file): 0
Skipped (no selected-class labels): 0

Converting test split...
Converted: 1500
Skipped (no image file): 0
Skipped (no selected-class labels): 0


In [10]:
for split in ['train', 'val', 'test']:
    path = os.path.join(LABEL_OUTPUT_DIR, split)
    if os.path.exists(path):
        print(f"{split.capitalize()} labels: {len(os.listdir(path))}")

Train labels: 7000
Val labels: 1500
Test labels: 1500


## Step 5: Organize into YOLO Dataset Structure

In [11]:
import shutil

from src.convert_labels import _find_bdd100k_image

_bdd_copy_roots = [os.path.abspath(TRAIN_IMAGE_DIR), os.path.abspath(VAL_IMAGE_DIR)]

for split_name, items in [("train", train_items), ("val", val_items), ("test", test_items)]:
    img_dst = os.path.join(OUTPUT_DIR, "images", split_name)
    lbl_dst = os.path.join(OUTPUT_DIR, "labels", split_name)
    os.makedirs(img_dst, exist_ok=True)
    os.makedirs(lbl_dst, exist_ok=True)

    paired_count = 0
    for item in items:
        img_name = item["name"]
        src_img = _find_bdd100k_image(img_name, _bdd_copy_roots)
        if not src_img:
            continue
        disk_basename = os.path.basename(src_img)
        stem, _ = os.path.splitext(disk_basename)
        label_name = stem + ".txt"
        src_label = os.path.join(LABEL_OUTPUT_DIR, split_name, label_name)

        if os.path.exists(src_label):
            shutil.copy2(src_img, os.path.join(img_dst, disk_basename))
            shutil.copy2(src_label, os.path.join(lbl_dst, label_name))
            paired_count += 1

    print(f"{split_name}: successfully copied {paired_count} image/label pairs")

train: successfully copied 7000 image/label pairs
val: successfully copied 1500 image/label pairs
test: successfully copied 1500 image/label pairs


## Step 6: Post-Conversion Visual Verification

In [12]:
from src.convert_labels import visual_verification

train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
train_lbl_dir = os.path.join(OUTPUT_DIR, "labels", "train")

train_images = glob.glob(os.path.join(train_img_dir, "*.jpg"))
print(f"Found {len(train_images)} training images for verification")

visual_verification(
    image_paths=train_images,
    label_dir=train_lbl_dir,
    output_path=os.path.join(RESULTS_PLOTS, "conversion_verification.png"),
    grid_size=3,
    num_samples=9,
)

Found 7000 training images for verification
Verification grid saved to C:\Users\micha\Downloads\Object-Detection-main\outputs\bdd100k_preprocessing\results\plots\conversion_verification.png


## Step 7: Summary & optional zip

In [13]:
for split_name in ["train", "val", "test"]:
    img_count = len(glob.glob(os.path.join(OUTPUT_DIR, "images", split_name, "*.jpg")))
    lbl_count = len(glob.glob(os.path.join(OUTPUT_DIR, "labels", split_name, "*.txt")))
    print(f"{split_name}: {img_count} images, {lbl_count} labels")

print("\nDataset ready at:", OUTPUT_DIR)

train: 7000 images, 7000 labels
val: 1500 images, 1500 labels
test: 1500 images, 1500 labels

Dataset ready at: C:\Users\micha\Downloads\Object-Detection-main\outputs\bdd100k_preprocessing\bdd100k-yolo-subset-v1


In [14]:
import zipfile
import os

# Optional: archive YOLO subset + split JSONs for upload or backup
output_zip = os.path.join(WORKING_ROOT, "bdd100k_yolo_labels.zip")
output_dir = OUTPUT_DIR
extra_files = [
    os.path.join(WORKING_ROOT, "test_annotations.json"),
    os.path.join(WORKING_ROOT, "train_annotations.json"),
    os.path.join(WORKING_ROOT, "val_annotations.json"),
]

with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(output_dir):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, start=output_dir)
            zipf.write(file_path, arcname=os.path.join(os.path.basename(output_dir), arcname))
    
    for file in extra_files:
        if os.path.isfile(file):
            zipf.write(file, os.path.basename(file))

print("Wrote:", output_zip)

Wrote: C:\Users\micha\Downloads\Object-Detection-main\outputs\bdd100k_preprocessing\bdd100k_yolo_labels.zip
